# Read Table -> Establish Generic Silver-level Table

In [0]:
#Imports to run API Data Scraping script

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import logging
import traceback
import json
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
#parameters that need to be changed between notebooks

silver_level_schema = StructType([
    StructField('account_open_date', StringType(), True),
    StructField('account_open_year', StringType(), True),
    StructField('account_open_month', StringType(), True),
    StructField('expiry_date', StringType(), True),
    StructField('expiry_year', StringType(), True),
    StructField('expiry_month', StringType(), True),
    StructField('year_pin_last_changed', StringType(), True),
    StructField('id', StringType(), True),
    StructField('client_id', StringType(), True),
    StructField('card_brand', StringType(), True),
    StructField('card_type', StringType(), True),
    StructField('card_number', StringType(), True),
    StructField('expires', DateType(), True),
    StructField('cvv', StringType(), True),
    StructField('has_chip', StringType(), True),
    StructField('num_cards_issued', IntegerType(), True),
    StructField('credit_limit', IntegerType(), True),
    StructField('acct_open_date', DateType(), True),
    StructField('card_on_dark_web', StringType(), True)
])

schema = silver_level_schema
schema_path = 'financial_transactions_dataset.default'
volume_path = '/Volumes/financial_transactions_dataset/default/financial_transactions_dataset_analytics_volume'
volume_identifier = 'cards_data'
silver_table_name = 'silver_cards_data'
bronze_table_name = 'bronze_cards_data'

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/silver_level_table_run_{datetime.now().strftime('%Y%m%d')}.log'),
                    logging.StreamHandler(sys.stdout)
                   ]
    )

    return logging.getLogger(__name__)

In [0]:
#Check if a table exists in a schema; create an empty table if it does not exist.

def check_or_create_table(logger, silver_table_name, schema_path):

    logger.info(f'Checking to see if {silver_table_name} exists.')
    
    if spark.catalog.tableExists(f'{schema_path}.{silver_table_name}'):
        logger.info("Table exists; table creation skipped.")
        flag = 1
    else:
        logger.info('Table does not exist; table will be created.')
        flag = 0
    
    return flag


In [0]:
#Pull data from bronze-level preliminary table

    #Function to read a SQL data table as a dataframe

def read_table_as_df(logger, schema_path, bronze_table_name):

    logger.info(f'Reading data from {schema_path}.{bronze_table_name}')

    df = spark.read.table(f'{schema_path}.{bronze_table_name}')
    total_row_count = df.count()

    logger.info(f'Read {total_row_count} rows from {schema_path}.{bronze_table_name}')
    
    return df


In [0]:
def standardize_date_fields(logger, df):
    logger.info(f'Reformatting expiration date')

    df = df.withColumn('expiry_month', F.split(F.col('expires'), '/').getItem(0))
    df = df.withColumn('expiry_year', F.split(F.col('expires'), '/').getItem(1))
    df = df.withColumn('expiry_date', F.concat_ws('-',F.split(F.col('expires'), '/').getItem(1), F.split(F.col('expires'), '/').getItem(0)))   

    logger.info(f'Reformatting account open date')

    df = df.withColumn('account_open_month', F.split(F.col('acct_open_date'), '/').getItem(0))
    df = df.withColumn('account_open_year', F.split(F.col('acct_open_date'), '/').getItem(1))
    df = df.withColumn('account_open_date', F.concat_ws('-',F.split(F.col('acct_open_date'), '/').getItem(1),F.split(F.col('acct_open_date'), '/').getItem(0)))

    return df


In [0]:
def drop_excess_date_fields(logger, df):
    logger.info(f'Dropping original expiration, account open dates')

    df = df.drop('expires', 'acct_open_date')

    return df


In [0]:
def convert_to_date_type(logger, df):
    logger.info(f'Converting expiration and account open dates to ''yyyy-MM'' format')
    
    df = df.withColumn('expiry_date', F.date_format(F.col('expiry_date'), 'yyyy-MM'))
    df = df.withColumn('account_open_date', F.date_format(F.col('account_open_date'), 'yyyy-MM'))

    return df


In [0]:
def capitalize_field_values(logger, df):
    logger.info(f'Capitalizing values in card_on_dark_web')
    
    df = df.withColumn('card_on_dark_web', F.upper(F.col('card_on_dark_web')))

    return df

In [0]:
def reorder_fields(logger, df):
    logger.info(f'Reordering arrangement of fields')

    df = df.select(
        'account_open_date',
        'account_open_year',
        'account_open_month',
        'expiry_date',
        'expiry_year',
        'expiry_month',
        'year_pin_last_changed',
        'card_id',
        'client_id',
        'card_brand',
        'card_type',
        'card_number',
        'cvv',
        'has_chip',
        'num_cards_issued',
        'credit_limit',
        'card_on_dark_web'
    )
    return df

In [0]:
#Function to overwrite/append data to silver level table

def create_or_append_to_table(logger, table_exists, df, schema_path, silver_table_name):

    if table_exists == 1:
        logger.info(f'Appending data to {schema_path}')

        df.write.format('delta').mode('append').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    else:

        logger.info(f'Creating table to {schema_path}')

        df.write.format('delta').mode('overwrite').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    return


In [0]:
def credit_limit_to_int(logger, df):
    logger.info(f'Converting credit limit to integer')

    df = df.withColumn('credit_limit', F.regexp_replace(F.col('credit_limit'), '\\$', ''))
    df = df.withColumn('credit_limit', F.col('credit_limit').cast('int'))

    return df


In [0]:
def rename_field(logger, df):
    logger.info(f'Renaming id field to card_id.')

    df = df.withColumnRenamed('id', 'card_id')

    return df

In [0]:
def silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier):

    schema = silver_level_schema
    schema_path = schema_path
    volume_path = volume_path
    volume_identifier = volume_identifier
    silver_table_name = silver_table_name
    bronze_table_name = bronze_table_name
    
    logger = setup_logging()

    logger.info(f'Initializing ETL on bronze dataframe')

    is_created = check_or_create_table(logger, silver_table_name, schema_path)
    bronze_df = read_table_as_df(logger, schema_path, bronze_table_name)
    date_df = standardize_date_fields(logger, bronze_df)
    excess_df = drop_excess_date_fields(logger, date_df)
    formatted_df = convert_to_date_type(logger, excess_df)
    cap_df = capitalize_field_values(logger, formatted_df)
    credit_df = credit_limit_to_int(logger, cap_df)
    renamed_df = rename_field(logger, credit_df)
    silver_df = reorder_fields(logger, renamed_df)

    create_or_append_to_table(logger, is_created, silver_df, schema_path, silver_table_name)
    
    logger.info(f'Silver level table {silver_table_name} created or appended to successfully.')

    return


In [0]:
silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier)